# LeetCode 3512 — Minimum Operations to Make Array Sum Divisible by K bằng Python

Link đề: https://leetcode.com/problems/minimum-operations-to-make-array-sum-divisible-by-k/ · Độ khó: **Easy** · Chủ đề: Array, Math, Modulo

Skeleton LeetCode:

```python
class Solution:
    def minOperations(self, nums: List[int], k: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi thao tác **giảm một phần tử đi `1`**; tìm số thao tác **ít nhất** để **tổng mảng chia hết cho `k`**.
2. Nhận ra mỗi thao tác làm tổng giảm **đúng `1`**, nên bài chỉ phụ thuộc vào **tổng** (không phụ thuộc phần tử nào bị giảm) và đáp án là `tổng % k`.
3. Hiểu phép **modulo** (`%`): phần dư, tính chất `(a + b) % k = ((a % k) + (b % k)) % k`.
4. Viết 4 cách: mô phỏng từng thao tác, `sum % k`, cộng dồn modulo, và thử từng số thao tác `0..k-1`; đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: dùng `k - tổng % k` (trường hợp tăng), dùng chia nguyên `//`, cộng phần dư của **từng** phần tử mà không rút gọn, và quên trường hợp đã chia hết.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và số nguyên `k`. Bạn có thể thực hiện thao tác sau **nhiều lần tuỳ ý**: chọn một chỉ số `i` và thay `nums[i]` bằng `nums[i] - 1`.

Trả về **số thao tác tối thiểu** để **tổng** các phần tử của `nums` **chia hết cho `k`**.

| Ví dụ | nums, k | Output | Giải thích |
|---|---|---|---|
| 1 | `[3,9,7]`, `5` | 4 | tổng `19`, `19 % 5 = 4`; giảm tổng đi `4` (ví dụ `9 -> 5`) được `15` chia hết cho `5` |
| 2 | `[4,1,3]`, `4` | 0 | tổng `8` đã chia hết cho `4` |
| 3 | `[3,2]`, `6` | 5 | tổng `5`; muốn chia hết cho `6` phải về `0`: giảm `5` lần |

Constraints: `1 <= nums.length <= 1000`; `1 <= nums[i] <= 1000`; `1 <= k <= 100`.

In [1]:
nums, k = [3, 9, 7], 5
total = sum(nums)
print("tổng =", total, "| tổng % k =", total % k)
print("số thao tác tối thiểu =", total % k)

tổng = 19 | tổng % k = 4
số thao tác tối thiểu = 4


## 3. Kiến thức nền

### 3.1. Phép chia dư `%` (modulo)

`a % k` là **phần dư** khi chia `a` cho `k`, luôn nằm trong `[0, k - 1]` (với `k > 0`, kể cả `a` âm trong Python). `a` chia hết cho `k` khi và chỉ khi `a % k == 0`. Ví dụ `19 % 5 = 4` vì `19 = 3 * 5 + 4`.

In [2]:
for a in (19, 15, 8, 5, 0):
    print(f"{a} % 5 = {a % 5}")

19 % 5 = 4
15 % 5 = 0
8 % 5 = 3
5 % 5 = 0
0 % 5 = 0


### 3.2. Giảm tổng đi `d` thì phần dư thay đổi thế nào

Nếu `S % k = r` thì `S = q * k + r`. Giảm tổng đi `r` được `q * k` — chia hết. Giảm ít hơn `r` (`d < r`) cho `S - d = q * k + (r - d)` có dư `r - d > 0` — chưa chia hết. Vậy muốn dư về `0` bằng cách **chỉ giảm**, ta cần giảm **ít nhất `r`**. (Giảm `r + k` cũng chia hết nhưng tốn hơn.)

In [3]:
S, k = 19, 5
for d in range(0, 8):
    print(f"giảm {d}: tổng {S - d}, dư {(S - d) % k}")

giảm 0: tổng 19, dư 4
giảm 1: tổng 18, dư 3
giảm 2: tổng 17, dư 2
giảm 3: tổng 16, dư 1
giảm 4: tổng 15, dư 0
giảm 5: tổng 14, dư 4
giảm 6: tổng 13, dư 3
giảm 7: tổng 12, dư 2


### 3.3. `sum()` và tính chất cộng của modulo

`sum(nums)` cộng mọi phần tử. Với modulo: `(a + b) % k == ((a % k) + (b % k)) % k`. Nhờ đó có thể cộng dồn **trong khi lấy dư** để số luôn nhỏ (hữu ích ở C++/Java khi số lớn; Python thì không tràn).

In [4]:
nums, k = [3, 9, 7], 5
print(sum(nums) % k)
r = 0
for x in nums:
    r = (r + x) % k
    print(f"cộng {x}: dư hiện tại = {r}")

4
cộng 3: dư hiện tại = 3
cộng 9: dư hiện tại = 2
cộng 7: dư hiện tại = 4


### 3.4. Tổng của các dư **chưa rút gọn** khác dư của tổng

`sum(x % k for x in nums)` có thể **lớn hơn** `k - 1`. Ví dụ `[3, 9, 7]`, `k = 5`: dư từng phần tử `3, 4, 2` có tổng `9`, nhưng dư của tổng `19` là `4`. Muốn đúng phải lấy `% k` thêm một lần nữa: `9 % 5 = 4`.

In [5]:
nums, k = [3, 9, 7], 5
parts = [x % k for x in nums]
print(parts, sum(parts), sum(parts) % k, sum(nums) % k)

[3, 4, 2] 9 4 4


### 3.5. Vòng lặp `for ops in range(k)` thử đáp án

Vì đáp án nằm trong `[0, k - 1]`, ta có thể **thử** lần lượt `ops = 0, 1, ..., k - 1` và dừng ở giá trị đầu tiên mà `(tổng - ops) % k == 0`. Cách này chậm hơn công thức nhưng **chứng minh trực tiếp** rằng đáp án là phần dư.

In [6]:
S, k = 19, 5
for ops in range(k):
    ok = (S - ops) % k == 0
    print(f"ops={ops}: tổng còn {S - ops} chia hết cho {k}? {ok}")
    if ok:
        break

ops=0: tổng còn 19 chia hết cho 5? False
ops=1: tổng còn 18 chia hết cho 5? False
ops=2: tổng còn 17 chia hết cho 5? False
ops=3: tổng còn 16 chia hết cho 5? False
ops=4: tổng còn 15 chia hết cho 5? True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Mỗi thao tác chỉ làm **tổng** giảm `1` bất kể chọn phần tử nào. Nên bài toán tương đương: "tổng `S` hiện tại; mỗi bước `S -= 1`; sau bao nhiêu bước thì `S` chia hết cho `k`?"

```
nums = [3, 9, 7], k = 5, S = 19
S = 19  (dư 4) -> thao tác 1 -> 18 (dư 3) -> 17 (dư 2) -> 16 (dư 1) -> 15 (dư 0)
```

`4` thao tác: bằng đúng **phần dư** `19 % 5 = 4`. Ví dụ 3: `S = 5`, `k = 6`: `5 % 6 = 5` -> `5` thao tác (`5 -> 0`, `0` chia hết cho `6`). Tổng không bị chặn dưới nên **được phép** giảm về `0` (hoặc thậm chí âm).

## 5. Cách 1 — Brute Force / baseline

Mô phỏng nguyên văn: bản sao `work`; trong khi `sum(work) % k != 0`: chọn một phần tử (ví dụ phần tử đầu) và giảm `1`, đếm thao tác.

Nút thắt: mỗi vòng gọi `sum(work)` tốn `O(n)`; số vòng tối đa `k - 1`, tổng `O(n * k)`. Chạy tốt cho `n <= 1000`, `k <= 100` nhưng thừa thãi: ta đã biết số vòng là phần dư. Cũng cần lưu ý: phần tử có thể xuống `0` hoặc âm khi bị giảm nhiều (đề không cấm), nên không có lo ngại "hết phần tử để giảm".

In [7]:
class SolutionSimulate:
    def minOperations(self, nums, k):
        work = list(nums)
        ops = 0
        while sum(work) % k != 0:
            work[0] -= 1
            ops += 1
        return ops

print(SolutionSimulate().minOperations([3, 9, 7], 5))
print(SolutionSimulate().minOperations([4, 1, 3], 4))
print(SolutionSimulate().minOperations([3, 2], 6))

4
0
5


**Độ phức tạp:** Time `O(n * k)`, Space `O(n)` cho bản sao.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** số thao tác **không phụ thuộc** phần tử nào bị giảm; chỉ cần biết tổng `S` và `r = S % k`. Cần đúng `r` thao tác: ít hơn thì dư còn `> 0`, còn `r` thao tác thì `S - r` chia hết cho `k`.

**Quy tắc "nếu… thì…":**

- Nếu `S % k == 0` -> đáp án `0` (đã chia hết).
- Ngược lại -> đáp án `S % k`. Hai trường hợp gộp thành một công thức `S % k` vì `0 % k = 0`.

Invariant: sau `t` thao tác (`t <= r`), dư của tổng là `r - t`; dư về `0` lần đầu khi `t = r`.

In [8]:
def formula(nums, k):
    return sum(nums) % k

print(formula([3, 9, 7], 5), formula([4, 1, 3], 4), formula([3, 2], 6))

4 0 5


## 7. Các cách tiếp theo

Ta có 4 cách: (1) mô phỏng từng thao tác; (2) `sum(nums) % k`; (3) cộng dồn modulo (không tạo tổng lớn); (4) thử `ops = 0..k-1`.

### Cách 2 — `sum(nums) % k`

Một dòng: `O(n)` để tính tổng, rồi một phép `%`.

In [9]:
class SolutionModSum:
    def minOperations(self, nums, k):
        return sum(nums) % k

print(SolutionModSum().minOperations([3, 9, 7], 5))
print(SolutionModSum().minOperations([4, 1, 3], 4))
print(SolutionModSum().minOperations([3, 2], 6))

4
0
5


Độ phức tạp Cách 2: Time `O(n)`, Space `O(1)`.

### Cách 3 — Cộng dồn modulo

Giữ `r` luôn trong `[0, k - 1]`: `r = (r + x) % k` với từng `x`. Cuối cùng `r` chính là `S % k`. Ưu điểm: số không bao giờ lớn (quan trọng ở ngôn ngữ có số nguyên cố định, hoặc khi dữ liệu streaming).

In [10]:
class SolutionRunningMod:
    def minOperations(self, nums, k):
        r = 0
        for x in nums:
            r = (r + x) % k
        return r

print(SolutionRunningMod().minOperations([3, 9, 7], 5))
print(SolutionRunningMod().minOperations([4, 1, 3], 4))
print(SolutionRunningMod().minOperations([3, 2], 6))

4
0
5


Độ phức tạp Cách 3: Time `O(n)`, Space `O(1)`.

### Cách 4 — Thử đáp án `0..k-1`

Tính `S = sum(nums)`; với `ops` từ `0` tăng dần, trả về `ops` đầu tiên có `(S - ops) % k == 0`. Vì trong `k` số nguyên liên tiếp luôn có một số chia hết cho `k`, vòng lặp chắc chắn dừng trước `k`.

In [11]:
class SolutionTry:
    def minOperations(self, nums, k):
        S = sum(nums)
        for ops in range(k):
            if (S - ops) % k == 0:
                return ops

print(SolutionTry().minOperations([3, 9, 7], 5))
print(SolutionTry().minOperations([4, 1, 3], 4))
print(SolutionTry().minOperations([3, 2], 6))

4
0
5


Độ phức tạp Cách 4: Time `O(n + k)`, Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 3 (cộng dồn modulo) với `nums = [3, 9, 7]`, `k = 5`:

| bước | `x` | `r` trước | `r + x` | `(r + x) % 5` |
|---|---|---|---|---|
| 1 | 3 | 0 | 3 | 3 |
| 2 | 9 | 3 | 12 | 2 |
| 3 | 7 | 2 | 9 | 4 |

Kết quả `r = 4`: cần `4` thao tác.

In [12]:
def trace_mod(nums, k):
    r = 0
    for step, x in enumerate(nums, start=1):
        before = r
        r = (r + x) % k
        print(f"bước {step}: x={x} r_trước={before} r+x={before + x} -> (r+x) % {k} = {r}")
    print(f"tổng = {sum(nums)}, tổng % {k} = {sum(nums) % k}; cần {r} thao tác")
    return r

trace_mod([3, 9, 7], 5)
trace_mod([4, 1, 3], 4)

bước 1: x=3 r_trước=0 r+x=3 -> (r+x) % 5 = 3
bước 2: x=9 r_trước=3 r+x=12 -> (r+x) % 5 = 2
bước 3: x=7 r_trước=2 r+x=9 -> (r+x) % 5 = 4
tổng = 19, tổng % 5 = 4; cần 4 thao tác
bước 1: x=4 r_trước=0 r+x=4 -> (r+x) % 4 = 0
bước 2: x=1 r_trước=0 r+x=1 -> (r+x) % 4 = 1
bước 3: x=3 r_trước=1 r+x=4 -> (r+x) % 4 = 0
tổng = 8, tổng % 4 = 0; cần 0 thao tác


0

## 9. Code nộp LeetCode

In [13]:
from typing import List

class Solution:
    def minOperations(self, nums: List[int], k: int) -> int:
        return sum(nums) % k

print(Solution().minOperations([3, 9, 7], 5), Solution().minOperations([4, 1, 3], 4), Solution().minOperations([3, 2], 6))

4 0 5


## 10. Vì sao thuật toán đúng

- **Mỗi thao tác đổi tổng đúng `-1`:** thay `nums[i]` bằng `nums[i] - 1` chỉ đổi một số hạng đi `1`, nên sau `t` thao tác tổng là `S - t` bất kể chọn `i` nào.
- **Cần ít nhất `r` thao tác:** với `t` thao tác ta có tổng `S - t`. Viết `S = q * k + r` (`0 <= r < k`). Với `0 <= t < r`: `S - t = q * k + (r - t)` và `0 < r - t < k`, **không** chia hết cho `k`. Nên chưa thể thành công với `t < r`.
- **Đủ `r` thao tác:** với `t = r`, `S - r = q * k` chia hết cho `k`. Và chọn bất kỳ `r` lần giảm (ví dụ giảm `nums[0]` `r` lần) đều hợp lệ vì đề không giới hạn giá trị phần tử về mặt dưới.
- Trường hợp `r = 0` cho `0` thao tác, khớp công thức `S % k`.
- **Cách 3:** do `(a + b) % k = ((a % k) + (b % k)) % k`, việc cộng dồn modulo cho đúng `S % k`. **Cách 4:** vòng lặp tìm số `ops` nhỏ nhất làm `(S - ops) % k = 0` và theo phân tích trên chính là `r`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng từng thao tác | `O(n * k)` | `O(n)` |
| 2. `sum(nums) % k` | `O(n)` | `O(1)` |
| 3. Cộng dồn modulo | `O(n)` | `O(1)` |
| 4. Thử `ops = 0..k-1` | `O(n + k)` | `O(1)` |

Cách 2 và 3 là tối ưu (phải đọc hết `n` phần tử ít nhất một lần).

## 12. Bộ test

Test 1: ví dụ đề và edge case (`n = 1`, đã chia hết, `k = 1`, tổng nhỏ hơn `k`, tổng đúng bằng bội của `k`, biên lớn).

In [14]:
impls = [SolutionSimulate(), SolutionModSum(), SolutionRunningMod(), SolutionTry(), Solution()]
tests = [
    ([3, 9, 7], 5, 4),
    ([4, 1, 3], 4, 0),
    ([3, 2], 6, 5),
    ([7], 1, 0),                       # k = 1: mọi tổng đều chia hết
    ([1], 100, 1),                     # tổng nhỏ hơn k
    ([100], 100, 0),                   # tổng đúng bằng k
    ([99], 100, 99),                   # gần bội kế tiếp nhưng phải GIẢM, không tăng
    ([1, 1, 1], 3, 0),                 # tổng 3 chia hết cho 3
    ([1, 1, 1, 1], 3, 1),              # tổng 4 -> dư 1
    ([1000] * 1000, 100, 0),           # biên: tổng 1_000_000 chia hết
    ([1000] * 999 + [999], 100, 99),   # tổng 999_999 -> dư 99
    ([5, 5, 5], 7, 1),                 # tổng 15 -> dư 1
]
for nums, k, exp in tests:
    assert 1 <= len(nums) <= 1000 and all(1 <= v <= 1000 for v in nums) and 1 <= k <= 100
    for im in impls:
        got = im.minOperations(list(nums), k)
        assert got == exp, (type(im).__name__, nums[:5], k, got, exp)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 12


Test 2: ngẫu nhiên có seed, đúng constraints. Oracle độc lập: **tìm kiếm** số thao tác nhỏ nhất theo định nghĩa bằng cách giảm tổng từng đơn vị (không dùng `%` trên mảng mà dùng `divmod` trên tổng đã trừ).

In [15]:
import random
random.seed(3512)

def oracle(nums, k):
    s = sum(nums)
    ops = 0
    while divmod(s, k)[1] != 0:
        s -= 1
        ops += 1
    return ops

for t in range(1500):
    n = random.randint(1, 1000) if t % 100 == 0 else random.randint(1, 12)
    hi_val = random.choice([5, 50, 1000])
    k = random.randint(1, 100)
    nums = [random.randint(1, hi_val) for _ in range(n)]
    assert 1 <= n <= 1000 and all(1 <= v <= 1000 for v in nums) and 1 <= k <= 100
    exp = oracle(nums, k)
    for im in impls:
        assert im.minOperations(list(nums), k) == exp, (type(im).__name__, nums[:5], k)
print("1500 test ngẫu nhiên OK")

1500 test ngẫu nhiên OK


Test 3: tính chất — không sửa input; `0 <= đáp án < k`; `(sum - đáp án) % k == 0`; không phụ thuộc thứ tự (xáo trộn); với `k = 1` luôn `0`; đáp án `0` khi tổng chia hết `k`.

In [16]:
random.seed(512)
for _ in range(500):
    n = random.randint(1, 50)
    nums = [random.randint(1, 1000) for _ in range(n)]
    k = random.randint(1, 100)
    snap = list(nums)
    ans = Solution().minOperations(nums, k)
    assert nums == snap
    assert 0 <= ans < k
    assert (sum(nums) - ans) % k == 0
    sh = list(nums)
    random.shuffle(sh)
    assert Solution().minOperations(sh, k) == ans
    assert Solution().minOperations(nums, 1) == 0
    if sum(nums) % k == 0:
        assert ans == 0
print("tính chất (0 <= ans < k, chia hết sau khi giảm, xáo trộn): OK")

tính chất (0 <= ans < k, chia hết sau khi giảm, xáo trộn): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: nhầm thành bài "tăng" — trả về `k - tổng % k`

In [17]:
def wrong_increase(nums, k):
    return k - sum(nums) % k                    # SAI: đó là số lần TĂNG 1 để lên bội kế tiếp

nums, k = [3, 9, 7], 5
print("sai :", wrong_increase(nums, k))
print("đúng:", Solution().minOperations(nums, k))
assert wrong_increase(nums, k) != Solution().minOperations(nums, k)

sai : 1
đúng: 4


Vì sao sai: đề chỉ cho **giảm** `1` mỗi lần. Giảm `19 -> 15` tốn `4`; còn `k - 4 = 1` là số bước **tăng** `19 -> 20`. **Sửa:** `sum(nums) % k`.

### Lỗi 2: chia nguyên `//` thay vì lấy dư `%`

In [18]:
def wrong_floor_div(nums, k):
    return sum(nums) // k                       # SAI: đây là thương, không phải phần dư

nums, k = [3, 9, 7], 5
print("sai :", wrong_floor_div(nums, k))
print("đúng:", Solution().minOperations(nums, k))
assert wrong_floor_div(nums, k) != Solution().minOperations(nums, k)

sai : 3
đúng: 4


Vì sao sai: `19 // 5 = 3` là "bao nhiêu nhóm `5` trong `19`", còn số bước cần giảm là phần dư `19 % 5 = 4`. **Sửa:** dùng `%`.

### Lỗi 3: cộng phần dư của từng phần tử mà không rút gọn

In [19]:
def wrong_sum_of_mods(nums, k):
    return sum(x % k for x in nums)             # SAI: tổng các dư có thể >= k

nums, k = [3, 9, 7], 5
print("sai :", wrong_sum_of_mods(nums, k))
print("đúng:", Solution().minOperations(nums, k))
assert wrong_sum_of_mods(nums, k) != Solution().minOperations(nums, k)

sai : 9
đúng: 4


Vì sao sai: dư từng phần tử là `3, 4, 2`, tổng `9` — lớn hơn `k - 1 = 4`, và phải thêm một lần `% 5` mới ra `4`. **Sửa:** `sum(x % k for x in nums) % k` hoặc `sum(nums) % k`.

### Lỗi 4: dùng `k - tổng % k` mà không xét trường hợp đã chia hết

In [20]:
def wrong_when_divisible(nums, k):
    r = sum(nums) % k
    return k - r if r < k else 0                # SAI: khi r = 0 trả về k thay vì 0

nums, k = [4, 1, 3], 4
print("sai :", wrong_when_divisible(nums, k))
print("đúng:", Solution().minOperations(nums, k))
assert wrong_when_divisible(nums, k) != Solution().minOperations(nums, k)

sai : 4
đúng: 0


Vì sao sai: tổng `8` đã chia hết cho `4` nhưng công thức "k - dư" trả `4 - 0 = 4`. Luôn kiểm tra trường hợp dư `= 0` (đáp án `0`). Công thức `sum(nums) % k` tự xử lý được.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Làm cho tổng chia hết cho `k`" + thao tác thay đổi tổng đúng `±1` -> đáp án liên quan đến **`tổng % k`**.
- Đọc kỹ thao tác là **giảm** hay **tăng**: giảm cần `r`, tăng cần `k - r` (nếu `r > 0`), cho phép cả hai thì `min(r, k - r)`.
- Khi mọi thao tác chỉ ảnh hưởng tổng, **không cần quan tâm** phần tử nào bị đổi.
- Modulo của tổng có thể tính **cộng dồn** để số luôn nhỏ.
- Kiểm tra: tổng nhỏ hơn `k`, tổng đã chia hết, `k = 1`.
- Phân biệt `//` (thương) và `%` (dư): bài này cần phần dư.

## 15. Biến thể của bài

1. **Được cả tăng và giảm `1`**: đáp án `min(r, k - r)`.
2. Thao tác **tăng** `1` (`nums[i] + 1`): đáp án `(k - r) % k`.
3. Thao tác thay `nums[i]` bằng `nums[i] - x` với `x` trong tập cho trước: trở thành bài đổi tiền (DP trên phần dư).
4. **Phần tử không được âm** (`nums[i] >= 0`): khi `r > sum(nums)` thì không thể (ở bài này `r <= sum` luôn nên vẫn đúng).
5. Muốn **tổng chia hết cho `k`** bằng cách **xoá** ít phần tử nhất (DP theo phần dư).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | giảm 1 phần tử tới khi tổng chia hết | `O(n * k)` | `O(n)` | hiểu đề, oracle |
| 2. `sum % k` | đáp án là phần dư | `O(n)` | `O(1)` | **chuẩn, nên nộp** |
| 3. Cộng dồn modulo | `r = (r + x) % k` | `O(n)` | `O(1)` | số lớn / streaming |
| 4. Thử `ops = 0..k-1` | tìm `ops` nhỏ nhất | `O(n + k)` | `O(1)` | chứng minh trực quan |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `minOperations([6, 8, 5], 7)` bằng bao nhiêu?

**Bài 2 (tính tay):** `nums = [10, 20, 33]`, `k = 9`: tổng, dư, số thao tác; sau khi giảm tổng bằng bao nhiêu?

**Bài 3 (điền TODO):** hoàn thành `running_mod(nums, k)` trả về dư của tổng bằng cộng dồn modulo.

**Bài 4 (tự viết lại):** viết `min_ops_both(nums, k)`: được **tăng hoặc giảm** `1`, trả về số thao tác tối thiểu `min(r, k - r)`.

**Bài 5 (mini challenge):** viết `final_array(nums, k)` trả về **một** mảng sau khi thực hiện đúng số thao tác tối thiểu (giảm các phần tử từ đầu, mỗi phần tử có thể giảm xuống thấp hơn `1`), sao cho tổng chia hết cho `k`; kiểm tra tổng chia hết và số bước đúng.

In [21]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().minOperations([6, 8, 5], 7))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def running_mod(nums, k):
    # TODO: r = 0; với mỗi x: r = (r + x) % k; trả về r
    return -1

5


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** mỗi bước giảm tổng đi `1`; số bước tối thiểu là phần dư của tổng khi chia cho `k`.

**Mức 2 (vừa):** Bài 3: cập nhật `r` sau mỗi `x`. Bài 4: tính `r = sum % k`, đáp án `min(r, k - r)` (khi `r = 0` ra `0`). Bài 5: `r = sum % k`; lặp `r` lần `out[i % ...]`, hoặc dồn hết vào `out[0] -= r`.

**Mức 3 (gần đáp án):** Bài 1: tổng `19`, `19 % 7 = 5` -> `5`. Bài 2: tổng `63`, `63 % 9 = 0` -> `0` (đã chia hết).

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1: 6 + 8 + 5 = 19 ; 19 % 7 = 5
assert Solution().minOperations([6, 8, 5], 7) == 5

# Bài 2: 10 + 20 + 33 = 63 ; 63 % 9 = 0
assert sum([10, 20, 33]) == 63
assert Solution().minOperations([10, 20, 33], 9) == 0

# Bài 3
def running_mod(nums, k):
    r = 0
    for x in nums:
        r = (r + x) % k
    return r
assert running_mod([3, 9, 7], 5) == 4
assert running_mod([4, 1, 3], 4) == 0
assert running_mod([1000] * 1000, 100) == 0

# Bài 4
def min_ops_both(nums, k):
    r = sum(nums) % k
    return min(r, k - r)
assert min_ops_both([3, 9, 7], 5) == 1          # 19 -> 20 chỉ cần tăng 1
assert min_ops_both([3, 9], 5) == 2             # 12: dư 2, k - r = 3
assert min_ops_both([4, 1, 3], 4) == 0

# Bài 5
def final_array(nums, k):
    out = list(nums)
    out[0] -= sum(nums) % k
    return out
for nums, k in (([3, 9, 7], 5), ([3, 2], 6), ([4, 1, 3], 4)):
    out = final_array(nums, k)
    assert sum(out) % k == 0
    assert sum(nums) - sum(out) == Solution().minOperations(nums, k)
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "thao tác đổi tổng đúng `-1`, cần chia hết cho `k`":**

```python
return sum(nums) % k            # giảm 1: cần đúng 'dư' thao tác
# nếu thao tác là TĂNG 1: return (k - sum(nums) % k) % k
```

**Checklist trước khi submit:**

- [ ] Thao tác là **giảm** `1` (đáp án `tổng % k`), không phải tăng?
- [ ] Dùng `%` (dư), không dùng `//` (thương)?
- [ ] Trường hợp đã chia hết cho `0` thao tác (không trả `k`)?
- [ ] Nếu cộng dư từng phần tử thì nhớ `% k` lần cuối?
- [ ] Đã thử `k = 1`, tổng nhỏ hơn `k`, tổng đúng bằng `k`?
- [ ] Chữ ký đúng `minOperations(self, nums: List[int], k: int) -> int`?